# Open-Meteo H2O: code snapshot
Run cells in order. Raw CSV is read-only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib, json, platform, sys

SOURCE_DIR = Path('/content/drive/MyDrive/Bigdata_H20')
assert (SOURCE_DIR / 'Data_OpenMeteo.zip').is_file(), f'Không thấy ZIP trong {SOURCE_DIR}. Kiểm tra đường dẫn.'
PROJECT = SOURCE_DIR / 'OpenMeteo_H2O_3_ThanhPho'
PROJECT.mkdir(exist_ok=True)
RUN_ID = datetime.now(ZoneInfo('Asia/Ho_Chi_Minh')).strftime('%Y%m%d_%H%M%S_%f')
RUN = PROJECT / 'runs' / RUN_ID
for name in ['code', 'logs', 'reports', 'figures', 'processed', 'features', 'splits', 'metrics', 'models', 'predictions', 'notebooks']:
    (RUN / name).mkdir(parents=True, exist_ok=True)
def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

ip = get_ipython()
# Gỡ hook của lần chạy trước để một ô chỉ ghi vào đúng lần chạy hiện tại.
for event, func in globals().get('_audit_hooks', []):
    ip.events.unregister(event, func)
_audit_counter = 0
_audit_current = None
def _audit_before(info):
    global _audit_counter, _audit_current
    _audit_counter += 1
    source = info.raw_cell
    digest = hashlib.sha256(source.encode()).hexdigest()
    path = RUN / 'code' / f'{_audit_counter:02d}_{digest[:10]}.py'
    path.write_text(source, encoding='utf-8')
    _audit_current = {'cell': _audit_counter, 'file': str(path), 'sha256': digest, 'started': datetime.now().isoformat()}
def _audit_after(result):
    if _audit_current is None:
        return
    entry = dict(_audit_current)
    entry['finished'] = datetime.now().isoformat()
    error = result.error_before_exec or result.error_in_exec
    entry['status'] = 'error' if error else 'ok'
    entry['error'] = str(error) if error else None
    with (RUN / 'logs' / 'cells.jsonl').open('a', encoding='utf-8') as f:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

_audit_hooks = [('pre_run_cell', _audit_before), ('post_run_cell', _audit_after)]
for event, func in _audit_hooks:
    ip.events.register(event, func)
# Lưu cả ô thiết lập đang chạy sau khi đã kết nối Drive.
setup_source = ip.history_manager.input_hist_raw[-1]
(RUN / 'code' / '00_setup.py').write_text(setup_source, encoding='utf-8')
save_json({'run_id': RUN_ID, 'source_dir': str(SOURCE_DIR), 'target': 'next_day_mean_us_aqi',
           'source': 'CAMS model output via Open-Meteo', 'timezone': 'Asia/Ho_Chi_Minh',
           'python': sys.version, 'created': datetime.now().isoformat()}, RUN / 'reports' / 'config.json')
print('Lưu mọi đầu ra tại:', RUN)

In [ ]:
import subprocess, io, zipfile, shutil
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'h2o==3.46.0.8'])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h2o
from h2o.automl import H2OAutoML, get_leaderboard
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
plt.rcParams.update({'figure.figsize': (12, 4), 'axes.grid': True, 'grid.alpha': 0.2})

CITY_MAP = {0: 'Ha Noi', 1: 'Da Nang', 2: 'TP.HCM'}
HISTORY_NAME = 'air_quality_3_cities_20230101_20261004.csv'
FORECAST_NAME = 'forecast_3_cities_20261005_20261009.csv'
raw_dir = PROJECT / 'raw'
raw_dir.mkdir(exist_ok=True)
manifest = []
def read_source_csv(payload):
    metadata = pd.read_csv(io.BytesIO(payload), nrows=3)
    table = pd.read_csv(io.BytesIO(payload), skiprows=5)
    assert set(metadata.location_id) == {0, 1, 2}
    assert (metadata.utc_offset_seconds == 25200).all(), 'Nguồn không dùng giờ Việt Nam.'
    assert set(table.location_id) == {0, 1, 2}
    return metadata, table

with zipfile.ZipFile(SOURCE_DIR / 'Data_OpenMeteo.zip') as z:
    for filename in [HISTORY_NAME, FORECAST_NAME]:
        members = [n for n in z.namelist() if Path(n).name == filename]
        assert len(members) == 1, f'Cần đúng một file {filename}, tìm được {members}'
        payload = z.read(members[0])
        digest = hashlib.sha256(payload).hexdigest()
        target = raw_dir / filename
        if target.exists():
            assert hashlib.sha256(target.read_bytes()).hexdigest() == digest, 'CSV raw đã khác ZIP; dừng để kiểm tra.'
        else:
            target.write_bytes(payload)  # Sao chép nguyên byte; không ghi lại bằng pandas.
        manifest.append({'file': filename, 'sha256': digest, 'bytes': len(payload)})
history_meta, raw_history = read_source_csv((raw_dir / HISTORY_NAME).read_bytes())
forecast_meta, raw_forecast = read_source_csv((raw_dir / FORECAST_NAME).read_bytes())
save_json(manifest, RUN / 'reports' / 'raw_manifest.json')
save_json({'pandas': pd.__version__, 'numpy': np.__version__, 'h2o': h2o.__version__}, RUN / 'reports' / 'versions.json')
print('Lịch sử:', raw_history.shape, '| Dự báo riêng:', raw_forecast.shape)
display(history_meta, raw_history.head(), raw_forecast.head())

In [ ]:
def save_figure(name):
    plt.tight_layout()
    plt.savefig(RUN / 'figures' / name, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

raw_history.describe(include='all').to_csv(RUN / 'reports' / 'before_describe.csv')
missing_before = raw_history.isna().sum().rename('missing')
missing_before.to_csv(RUN / 'reports' / 'before_missing.csv')
display(raw_history.describe(), missing_before.to_frame())
preview = raw_history.assign(time=pd.to_datetime(raw_history.time), city=raw_history.location_id.map(CITY_MAP))
aqi_col = 'us_aqi (USAQI)'
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
missing_before.plot.bar(ax=axes[0], title='Before: missing values')
preview.boxplot(column=aqi_col, by='city', ax=axes[1])
axes[1].set_title('Before: AQI distribution by city')
fig.suptitle('')
save_figure('02_before_missing_boxplot.png')
for city, group in preview.groupby('city'):
    group.set_index('time')[aqi_col].resample('MS').mean().plot(label=city)
plt.title('Before: monthly mean AQI'); plt.ylabel('US AQI'); plt.legend()
save_figure('02_before_monthly_aqi.png')
for city, group in preview.groupby('city'):
    plt.hist(group[aqi_col].dropna(), bins=45, alpha=0.4, label=city)
plt.title('Before: hourly AQI histogram'); plt.legend()
save_figure('02_before_aqi_histogram.png')

In [ ]:
RENAME = {'pm10 (μg/m³)': 'pm10', 'pm2_5 (μg/m³)': 'pm2_5',
          'carbon_monoxide (μg/m³)': 'co', 'nitrogen_dioxide (μg/m³)': 'no2',
          'sulphur_dioxide (μg/m³)': 'so2', 'ozone (μg/m³)': 'o3', 'us_aqi (USAQI)': 'aqi'}
VALUES = list(RENAME.values())
def clean_hourly(raw):
    data = raw.rename(columns=RENAME).copy()
    assert all(c in data.columns for c in VALUES)
    data['time'] = pd.to_datetime(data['time'], errors='raise')
    data['city'] = data.location_id.map(CITY_MAP)
    assert data.city.notna().all()
    for col in VALUES:
        data[col] = pd.to_numeric(data[col], errors='coerce')
        data.loc[~np.isfinite(data[col]) | (data[col] < 0), col] = np.nan
    data = data.drop_duplicates()
    assert not data.duplicated(['city', 'time']).any(), 'Trùng mốc giờ nhưng khác giá trị.'
    assert (data.time.dt.minute == 0).all()
    data = data.sort_values(['city', 'time']).reset_index(drop=True)
    return data

hourly = clean_hourly(raw_history)
assert hourly.time.min() == pd.Timestamp('2023-01-01')
assert hourly.time.max() == pd.Timestamp('2026-10-04 23:00')
hourly.to_csv(RUN / 'processed' / 'hourly_clean.csv', index=False)
quality = []
for city, group in hourly.groupby('city'):
    expected = pd.date_range(group.time.min(), group.time.max(), freq='h')
    quality.append({'city': city, 'rows': len(group), 'missing_hours': len(expected.difference(group.time)),
                    'missing_values': int(group[VALUES].isna().sum().sum()),
                    'start': group.time.min(), 'end': group.time.max()})
quality = pd.DataFrame(quality)
quality.to_csv(RUN / 'reports' / 'hourly_quality.csv', index=False)
display(quality)

In [ ]:
def to_daily(data):
    temp = data.assign(date=data.time.dt.normalize())
    grouped = temp.groupby(['city', 'date'])
    means = grouped[VALUES].mean()
    counts = grouped[VALUES].count().add_suffix('_count')
    daily_table = means.join(counts)
    daily_table['hour_count'] = grouped.size()
    daily_table['aqi_max'] = grouped.aqi.max()
    daily_table['pm2_5_std'] = grouped.pm2_5.std()
    # Ngày thiếu giờ/thiếu AQI không được dùng làm nhãn; giữ dòng để lag không nhảy qua ngày thiếu.
    daily_table.loc[(daily_table.hour_count != 24) | (daily_table.aqi_count != 24), 'aqi'] = np.nan
    result = []
    for city, group in daily_table.reset_index().groupby('city'):
        group = group.set_index('date').reindex(pd.date_range(group.date.min(), group.date.max(), freq='D'))
        group.index.name = 'date'
        group['city'] = city
        result.append(group.reset_index())
    return pd.concat(result, ignore_index=True).sort_values(['city', 'date']).reset_index(drop=True)

daily = to_daily(hourly)
daily.to_csv(RUN / 'processed' / 'daily_clean.csv', index=False)
daily.describe(include='all').to_csv(RUN / 'reports' / 'after_daily_describe.csv')
display(daily.groupby('city')[VALUES].agg(['count', 'mean', 'std', 'min', 'max']))
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
daily[VALUES].isna().sum().plot.bar(ax=axes[0], title='After: daily missing values')
daily.boxplot(column='aqi', by='city', ax=axes[1]); axes[1].set_title('After: daily mean AQI')
fig.suptitle(''); save_figure('04_after_missing_boxplot.png')
for city, group in daily.groupby('city'):
    plt.plot(group.date, group.aqi, label=city, alpha=0.7, linewidth=0.8)
plt.title('After: daily mean AQI over time'); plt.ylabel('US AQI'); plt.legend()
save_figure('04_after_daily_aqi.png')
corr = daily[VALUES].corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(VALUES)), VALUES, rotation=45); ax.set_yticks(range(len(VALUES)), VALUES)
fig.colorbar(im, ax=ax); ax.set_title('After: daily feature correlation')
save_figure('04_after_correlation.png')

In [ ]:
def make_features(daily_table):
    parts = []
    for city, group in daily_table.groupby('city'):
        group = group.sort_values('date').copy()
        features = group[['city', 'date']].rename(columns={'date': 'input_date'})
        features['target_date'] = features.input_date + pd.Timedelta(days=1)
        for col in VALUES + ['aqi_max', 'pm2_5_std']:
            features[f'{col}_today'] = group[col].to_numpy()
        for lag in [1, 2, 3, 7]:
            features[f'aqi_lag_{lag}'] = group.aqi.shift(lag).to_numpy()
        for window in [3, 7]:
            features[f'aqi_mean_{window}'] = group.aqi.rolling(window, min_periods=window).mean().to_numpy()
            features[f'pm2_5_mean_{window}'] = group.pm2_5.rolling(window, min_periods=window).mean().to_numpy()
        features['target_aqi'] = group.aqi.shift(-1).to_numpy()
        features['input_complete'] = ((group.hour_count == 24) & (group.aqi_count == 24)).to_numpy()
        features['weekday'] = features.target_date.dt.dayofweek
        doy = features.target_date.dt.dayofyear
        features['year_sin'] = np.sin(2 * np.pi * doy / 365.25)
        features['year_cos'] = np.cos(2 * np.pi * doy / 365.25)
        parts.append(features)
    return pd.concat(parts, ignore_index=True).sort_values(['target_date', 'city']).reset_index(drop=True)

features_all = make_features(daily)
predictors = [c for c in features_all if c not in ['input_date', 'target_date', 'target_aqi', 'input_complete']]
dataset = features_all.loc[features_all.input_complete & features_all.target_aqi.notna() & features_all.aqi_lag_7.notna()].copy()
assert 'target_aqi' not in predictors and 'target_date' not in predictors
assert ((dataset.target_date - dataset.input_date).dt.days == 1).all()
# Kiểm tra hồi quy: thay giá trị tương lai không làm đổi đặc trưng của ngày trước đó.
probe = daily.copy(); cutoff = pd.Timestamp('2024-06-01')
probe.loc[probe.date > cutoff, VALUES] = 9999
original = features_all.loc[features_all.input_date <= cutoff, predictors].reset_index(drop=True)
changed = make_features(probe).loc[lambda d: d.input_date <= cutoff, predictors].reset_index(drop=True)
pd.testing.assert_frame_equal(original, changed)
dataset.to_csv(RUN / 'features' / 'supervised_next_day.csv', index=False)
save_json({'predictors': predictors, 'target': 'target_aqi', 'horizon_days': 1}, RUN / 'features' / 'schema.json')
display(dataset.head()); print('Dòng dùng train/eval:', len(dataset), '| Đặc trưng:', len(predictors))

In [ ]:
dates = np.array(sorted(dataset.target_date.unique()))
cuts = [0, int(len(dates)*0.6), int(len(dates)*0.7), int(len(dates)*0.8), int(len(dates)*0.9), len(dates)]
split_names = ['train', 'early_stop', 'blend', 'validation', 'test']
splits = {}; split_report = []
for i, name in enumerate(split_names):
    start = cuts[i] + (1 if i else 0)
    chosen_dates = dates[start:cuts[i+1]]
    part = dataset[dataset.target_date.isin(chosen_dates)].reset_index(drop=True)
    assert len(part) > 30 and part.city.nunique() == 3
    splits[name] = part
    part.to_csv(RUN / 'splits' / f'{name}.csv', index=False)
    split_report.append({'split': name, 'rows': len(part), 'input_start': part.input_date.min(),
                         'target_start': part.target_date.min(), 'target_end': part.target_date.max()})
for left, right in zip(split_names, split_names[1:]):
    assert splits[left].target_date.max() < splits[right].input_date.min()
split_report = pd.DataFrame(split_report)
split_report.to_csv(RUN / 'reports' / 'split_ranges.csv', index=False)
display(split_report)
plt.bar(split_report['split'], split_report.rows, color=['#146b77','#39a6a3','#78bbb2','#ecb34b','#d8745b'])
plt.title('Rows in chronological splits'); save_figure('06_split_sizes.png')

In [ ]:
def metrics(y, pred):
    y = np.asarray(y, dtype=float); pred = np.asarray(pred, dtype=float)
    assert len(y) == len(pred) and np.isfinite(y).all() and np.isfinite(pred).all()
    return {'mae': float(mean_absolute_error(y, pred)), 'rmse': float(np.sqrt(mean_squared_error(y, pred))), 'r2': float(r2_score(y, pred))}
baseline_rows = []
for name in ['validation', 'test']:
    frame = splits[name]
    baseline_rows.append({'split': name, 'model': 'Persistence: AQI tomorrow = today', **metrics(frame.target_aqi, frame.aqi_today)})
baseline_results = pd.DataFrame(baseline_rows)
baseline_results.to_csv(RUN / 'metrics' / 'baseline.csv', index=False)
display(baseline_results)

In [ ]:
import os
h2o.init(max_mem_size='3G', nthreads=2)
assert h2o.estimators.H2OXGBoostEstimator.available(), 'Runtime không hỗ trợ XGBoost; cần runtime Linux Colab.'
hframes = {}
for name, part in splits.items():
    hframes[name] = h2o.H2OFrame(part[predictors + ['target_aqi']])
    hframes[name]['city'] = hframes[name]['city'].asfactor()
aml = H2OAutoML(max_models=24, max_runtime_secs=0, max_runtime_secs_per_model=60,
               nfolds=0, seed=42, sort_metric='RMSE', stopping_metric='RMSE',
               project_name='openmeteo_nextday_' + RUN_ID,
               include_algos=None, exclude_algos=None,
               export_checkpoints_dir=str(RUN / 'models' / 'checkpoints'), verbosity='info')
save_json({'max_models': 24, 'max_runtime_secs_per_model': 60, 'nfolds': 0, 'seed': 42,
           'include_algos': None, 'exclude_algos': None, 'selection': 'validation RMSE',
           'blending': 'chronological dedicated split'}, RUN / 'reports' / 'automl_settings.json')
aml.train(x=predictors, y='target_aqi', training_frame=hframes['train'],
          validation_frame=hframes['early_stop'], blending_frame=hframes['blend'],
          leaderboard_frame=hframes['validation'])
leader = aml.leader
leaderboard = get_leaderboard(aml, extra_columns='ALL').as_data_frame()
leaderboard.to_csv(RUN / 'metrics' / 'leaderboard_validation.csv', index=False)
aml.event_log.as_data_frame().to_csv(RUN / 'logs' / 'automl_events.csv', index=False)
save_json(aml.modeling_steps, RUN / 'reports' / 'executed_modeling_steps.json')
def model_family(model_id):
    return model_id.split('_')[0]
leaderboard['family'] = leaderboard.model_id.map(model_family)
coverage = leaderboard.groupby('family').size().rename('models').reset_index()
coverage.to_csv(RUN / 'reports' / 'algorithm_coverage.csv', index=False)
display(coverage, leaderboard)
required = {'GLM','GBM','DRF','XRT','XGBoost','DeepLearning','StackedEnsemble'}
missing = sorted(required - set(coverage.family))
save_json({'required': sorted(required), 'missing': missing}, RUN / 'reports' / 'algorithm_coverage_check.json')
assert not missing, f'Có nhóm chưa tạo được model: {missing}. Xem automl_events.csv, không coi bản này đã đủ nhóm.'
print('Leader được chọn bằng validation:', leader.model_id)

In [ ]:
family_best = leaderboard.sort_values('rmse').drop_duplicates('family').copy()
family_best[['family','model_id','mae','rmse']].to_csv(RUN / 'metrics' / 'best_per_family_validation.csv', index=False)
test = splits['test']
test_rows = [{'family': 'Baseline', 'model_id': 'Persistence', **metrics(test.target_aqi, test.aqi_today)}]
test_predictions = test[['city','input_date','target_date','target_aqi','aqi_today']].copy()
for row in family_best.itertuples():
    model = h2o.get_model(row.model_id)
    pred = model.predict(hframes['test']).as_data_frame()['predict'].to_numpy()
    test_rows.append({'family': row.family, 'model_id': row.model_id, **metrics(test.target_aqi, pred)})
    if model.model_id == leader.model_id:
        test_predictions['prediction'] = pred
test_comparison = pd.DataFrame(test_rows)
test_comparison.to_csv(RUN / 'metrics' / 'family_comparison_test.csv', index=False)
test_predictions['error'] = test_predictions.prediction - test_predictions.target_aqi
test_predictions.to_csv(RUN / 'predictions' / 'test_predictions.csv', index=False)
leader_metrics = metrics(test.target_aqi, test_predictions.prediction)
save_json({'model_id': leader.model_id, 'test': leader_metrics,
           'baseline_test': metrics(test.target_aqi, test.aqi_today),
           'chosen_on': 'validation RMSE; test does not change leader'}, RUN / 'metrics' / 'leader_test.json')
city_metrics = pd.DataFrame([{'city': city, **metrics(g.target_aqi, g.prediction)} for city, g in test_predictions.groupby('city')])
city_metrics.to_csv(RUN / 'metrics' / 'test_by_city.csv', index=False)
display(family_best[['family','mae','rmse']], test_comparison, city_metrics)
print('Test chỉ chấm điểm; không chọn lại leader theo bảng Test.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
test_comparison.plot.bar(x='family', y=['mae','rmse'], ax=axes[0], title='Test: base families vs ensemble')
axes[1].scatter(test_predictions.target_aqi, test_predictions.prediction, s=10, alpha=0.4)
low = min(test_predictions.target_aqi.min(), test_predictions.prediction.min())
high = max(test_predictions.target_aqi.max(), test_predictions.prediction.max())
axes[1].plot([low,high],[low,high], '--', color='grey')
axes[1].set(xlabel='Reference CAMS AQI', ylabel='H2O predicted AQI', title='Selected leader: test predictions')
save_figure('10_test_comparison_scatter.png')
fig, axes = plt.subplots(3, 1, figsize=(14, 9))
for ax, (city, group) in zip(axes, test_predictions.groupby('city')):
    ax.plot(group.target_date, group.target_aqi, label='Reference CAMS', linewidth=1)
    ax.plot(group.target_date, group.prediction, label='H2O', linewidth=1)
    ax.plot(group.target_date, group.aqi_today, label='Baseline', alpha=0.4)
    ax.set_title(city); ax.legend()
save_figure('10_test_timeseries_cities.png')
plt.hist(test_predictions.error, bins=35); plt.title('Leader residuals on test'); plt.xlabel('Prediction - reference')
save_figure('10_test_residuals.png')
worst = test_predictions.assign(abs_error=lambda d: d.error.abs()).nlargest(20, 'abs_error')
worst.to_csv(RUN / 'reports' / 'worst_20_errors.csv', index=False)
display(worst)

In [ ]:
model_dir = RUN / 'models' / 'saved'
model_dir.mkdir(exist_ok=True)
saved_models = []
# Chỉ model thuộc lần AutoML này; kèm metalearner cần cho ensemble.
model_ids = set(leaderboard.model_id)
for model_id in list(model_ids):
    model = h2o.get_model(model_id)
    if model.algo == 'stackedensemble':
        model_ids.add(model.metalearner().model_id)
for model_id in sorted(model_ids):
    model = h2o.get_model(model_id)
    saved_models.append({'model_id': model_id, 'path': h2o.save_model(model, path=str(model_dir), force=True)})
save_json(saved_models, RUN / 'models' / 'saved_models.json')
mojo_path = leader.download_mojo(path=str(RUN / 'models'), get_genmodel_jar=True)
save_json({'leader': leader.model_id, 'mojo': mojo_path, 'h2o_version': h2o.__version__,
           'predictors': predictors, 'target': 'next_day_mean_us_aqi'}, RUN / 'models' / 'leader_manifest.json')
latest = features_all.loc[features_all.input_complete].sort_values('input_date').groupby('city').tail(1).copy()
hf_latest = h2o.H2OFrame(latest[predictors]); hf_latest['city'] = hf_latest['city'].asfactor()
latest['h2o_prediction'] = leader.predict(hf_latest).as_data_frame()['predict'].to_numpy()
next_day = latest[['city','input_date','target_date','h2o_prediction']].copy()
next_day['input_source'] = 'Completed historical CAMS day; not station observation'
next_day.to_csv(RUN / 'predictions' / 'next_day_from_completed_history.csv', index=False)
forecast_hourly = clean_hourly(raw_forecast)
forecast_daily = to_daily(forecast_hourly)
forecast_daily.to_csv(RUN / 'processed' / 'source_forecast_daily_SEPARATE.csv', index=False)
display(next_day, forecast_daily[['city','date','aqi']])
print('H2O dự đoán ngày sau ngày lịch sử cuối cùng. AQI 06/10 trong bảng nguồn dự báo là CAMS, không gọi là dự đoán H2O.')

In [ ]:
for item in manifest:
    assert hashlib.sha256((raw_dir / item['file']).read_bytes()).hexdigest() == item['sha256'], 'Raw CSV bị thay đổi!'
required_files = ['reports/raw_manifest.json', 'reports/split_ranges.csv', 'reports/algorithm_coverage.csv',
                  'processed/hourly_clean.csv', 'processed/daily_clean.csv', 'features/schema.json',
                  'metrics/leaderboard_validation.csv', 'metrics/family_comparison_test.csv',
                  'predictions/test_predictions.csv', 'models/leader_manifest.json']
assert all((RUN / p).is_file() for p in required_files)
code_files = sorted((RUN / 'code').glob('*.py'))
assert len(code_files) >= 13, f'Thiếu file code: chỉ có {len(code_files)}'
snapshot_cells = [{'cell_type':'markdown', 'metadata':{}, 'source':['# Open-Meteo H2O: code snapshot\n', 'Run cells in order. Raw CSV is read-only.']}]
for file in code_files:
    snapshot_cells.append({'cell_type':'code', 'metadata':{}, 'source':file.read_text(encoding='utf-8').splitlines(True), 'execution_count':None, 'outputs':[]})
save_json({'nbformat':4, 'nbformat_minor':5, 'metadata':{'kernelspec':{'name':'python3','display_name':'Python 3'}},
           'cells':snapshot_cells}, RUN / 'notebooks' / 'OpenMeteo_H2O_code_snapshot.ipynb')
summary = f"""# Open-Meteo H2O — 3 thành phố
Run: {RUN_ID}
Nguồn: CAMS qua Open-Meteo; dữ liệu mô hình, không phải số đo trạm.
Mục tiêu: AQI trung bình ngày kế tiếp; horizon 1 ngày.
Lịch sử: 01/01/2023 đến 04/10/2026. Forecast lưu riêng, không train/test.
Leader chọn trên validation: {leader.model_id}
Test MAE: {leader_metrics['mae']:.4f}; RMSE: {leader_metrics['rmse']:.4f}; R2: {leader_metrics['r2']:.4f}
Các nhóm thực sự đã train: {', '.join(coverage.family)}
So sánh baseline: metrics/baseline.csv. So sánh nhóm: metrics/family_comparison_test.csv.
Raw hash đã kiểm tra lại: nội dung CSV gốc không thay đổi.
Kết quả đo khả năng dự đoán chuỗi CAMS; không chứng minh độ chính xác so với không khí đo thực tế.
Đặc trưng cần ngày đầu vào hoàn tất. Muốn dự đoán 06/10 từ dữ liệu 05/10 phải cập nhật nguồn và xác định giờ chốt.
DeepLearning và giới hạn thời gian từng model có thể làm kết quả thay đổi nhẹ giữa các lần chạy.
"""
(RUN / 'reports' / 'SUMMARY.md').write_text(summary, encoding='utf-8')
save_json({'run_id':RUN_ID, 'path':str(RUN)}, PROJECT / 'LATEST_RUN.json')
files = pd.DataFrame([{'file':str(p.relative_to(RUN)), 'bytes':p.stat().st_size} for p in RUN.rglob('*') if p.is_file()])
files.to_csv(RUN / 'reports' / 'file_inventory.csv', index=False)
display(files.groupby(files.file.str.split('/').str[0]).agg(files=('file','count'), bytes=('bytes','sum')))
print('HOÀN TẤT. Code, biểu đồ, splits, metrics, models đều đã lưu Drive:', RUN)